# 02_synthval: Check the synthetic data
Reads the 4 files from Notebook 1 (`output/`). Run Notebook 1 first. After **Restart**, use **Run All**.

Order in every part: CODE → TEST → RUN.

## Setup

In [ ]:
import numpy as np, pandas as pd
import ipytest
ipytest.autoconfig()
print("Setup OK")

## Cell 1: Load the files from Notebook 1

In [ ]:
# CODE
# ---------------------------------------------------------------------------
# Cell 1: load the 4 files from Notebook 1
# ---------------------------------------------------------------------------
def load_outputs(folder="output"):
    patients = pd.read_csv(f"{folder}/patients.csv")
    relatives = pd.read_csv(f"{folder}/relatives.csv")
    visits = pd.read_csv(f"{folder}/visits.csv", parse_dates=["visit_date"])
    training = pd.read_csv(f"{folder}/training.csv", parse_dates=["index_date"])
    return patients, relatives, visits, training

In [ ]:
# RUN
PATIENTS, RELATIVES, VISITS, TRAINING = load_outputs("output")
for name, df in [("patients", PATIENTS), ("relatives", RELATIVES), ("visits", VISITS), ("training", TRAINING)]:
    print(f"{name}: {len(df):,} rows")

## Cell 2: Hard rules (must be 0 failures)
The 10 rules from the System Design, Section 7. Ranges come from the CDSS Data Dictionary and are typed here again on purpose, so a mistake in Notebook 1 cannot hide itself.

In [ ]:
# CODE
# ---------------------------------------------------------------------------
# Cell 2: the 10 hard rules (System Design, Section 7)
# Min/max copied from the CDSS Data Dictionary (oct31), Parameter Ingestion Tables.
# Written here again ON PURPOSE (not imported from Notebook 1), so a mistake in
# the generator's table cannot hide itself.
# ---------------------------------------------------------------------------
RANGES = {
    "age": (18, 100),
    # CVD
    "sbp": (70, 220), "dbp": (40, 130), "heart_rate": (40, 180), "bmi": (12.0, 60.0),
    "total_cholesterol": (100, 400), "hdl": (15, 120), "ldl": (30, 300), "triglycerides": (40, 1000),
    "hs_crp": (0.1, 50.0), "creatinine": (0.4, 10.0), "egfr": (5, 140), "lvef": (15, 75),
    "ctni": (0.01, 50.0), "nt_probnp": (10, 35000), "pack_years": (0, 120),
    "sodium_mg": (500, 10000), "activity_min_week": (0, 1050),
    # T2D
    "fpg": (50, 450), "pg_2h": (60, 600), "hba1c": (4.0, 15.0), "insulin": (1.0, 100.0),
    "c_peptide": (0.1, 12.0), "homa_ir": (0.2, 25.0), "waist_cm": (50, 160), "whr": (0.60, 1.40),
    "uacr": (1, 3500), "uric_acid": (1.5, 14.0), "carbs_g": (50, 600), "sedentary_hours": (1.0, 18.0),
    "gestational_diabetes": (0, 1), "acanthosis_nigricans": (0, 1),
    # CRD
    "fev1": (0.5, 5.5), "fvc": (1.0, 7.0), "fev1_fvc": (25.0, 95.0), "fev1_pct_pred": (15.0, 120.0),
    "pef": (50, 800), "eosinophils": (0, 2500), "feno": (5, 150), "spo2": (65.0, 100.0),
    "pao2": (40, 110), "paco2": (25, 90), "mmrc": (0, 4), "act": (5, 25), "cat": (0, 40),
    "ige": (2, 3000), "exacerbations_year": (0, 15), "occupational_dust": (0, 1),
    "allergic_rhinitis": (0, 1), "childhood_asthma": (0, 1),
    # NEURO
    "phq9": (0, 27), "gad7": (0, 21), "psqi": (0, 21), "sleep_hours": (1.0, 16.0),
    "alcohol_use": (0, 1), "tbi_history": (0, 1), "tsh": (0.01, 100.0), "vitamin_b12": (50, 2000),
    # FHRS: 0, no fixed maximum
    "fhrs_CVD": (0.0, np.inf), "fhrs_T2D": (0.0, np.inf), "fhrs_CRD": (0.0, np.inf), "fhrs_NEURO": (0.0, np.inf),
}

# FHRS, copied from Notebook 1 Part A (same FHRS Guide numbers)
FHRS_CFG = {"R": {1: 0.50, 2: 0.25, 3: 0.125}, "S": {"confirmed": 1.0, "not_sure": 0.5}, "cluster_bonus": 1.0,
            "penalties": {"CVD": {"secondhand_smoke": 0.15, "high_salt_diet": 0.15},
                          "T2D": {"processed_diet": 0.15, "low_home_activity": 0.15},
                          "CRD": {"secondhand_smoke": 0.15, "indoor_mold": 0.15, "coal_heating": 0.15},
                          "NEURO": {}}}
DOMAIN = {"CVD": ["CVD"], "T2D": ["T2D"], "CRD": ["asthma", "COPD"], "NEURO": ["depression", "GAD"]}

def onset_multiplier(disease, age, sex):
    if age is None: return 1.0
    if disease == "CVD":
        if age < 40: return 3.0
        if sex == "M": return 2.0 if age < 55 else 1.0 if age < 65 else 0.75
        return 2.0 if age < 65 else 1.0 if age < 75 else 0.75
    if disease in ("T2D", "COPD"):
        return 3.0 if age < 40 else 2.0 if age < 50 else 1.0 if age < 65 else 0.75
    if disease == "asthma":
        return 3.0 if age < 12 else 2.0 if age <= 25 else 1.0 if age <= 40 else 0.75
    if disease == "depression":
        return 3.0 if age < 18 else 2.0 if age <= 24 else 1.0 if age <= 40 else 0.75
    return 1.0

def fhrs(relatives, exposures, domain, on_year, cfg=FHRS_CFG):
    bio, confirmed_first = 0.0, {}
    for r in relatives:
        for c in r["conditions"]:
            if c["disease"] in DOMAIN[domain] and c["year"] <= on_year:
                s = cfg["S"]["confirmed"] if c["confirmed"] else cfg["S"]["not_sure"]
                bio += cfg["R"][r["degree"]] * onset_multiplier(c["disease"], c["age"], r["sex"]) * s
                if r["degree"] == 1 and c["confirmed"]:
                    confirmed_first[c["disease"]] = confirmed_first.get(c["disease"], 0) + 1
                break
    if any(n >= 2 for n in confirmed_first.values()):
        bio += cfg["cluster_bonus"]
    E = 1.0 + sum(v for k, v in cfg["penalties"][domain].items() if exposures.get(k))
    return round(bio * E, 2)

def egfr_ckd_epi(creatinine, age, sex):
    """CKD-EPI 2021 race-free equation; sex 'Male'/'Female'."""
    female = sex in ("Female", "F")
    k, a = (0.7, -0.241) if female else (0.9, -0.302)
    s = creatinine / k
    return 142 * min(s, 1) ** a * max(s, 1) ** -1.2 * 0.9938 ** age * (1.012 if female else 1)

def _family(relatives):
    """relatives.csv rows -> {patient_id: [relative dicts]} in the FHRS format."""
    fams = {}
    for (pid, _), g in relatives.groupby(["patient_id", "relative_no"], sort=False):
        first = g.iloc[0]
        conds = [{"disease": c.disease, "year": int(c.dx_year),
                  "age": None if pd.isna(c.dx_age) else int(c.dx_age),
                  "confirmed": str(c.confirmed) == "True"}
                 for c in g.itertuples() if pd.notna(c.disease)]
        fams.setdefault(pid, []).append({"degree": int(first.degree), "sex": first.sex, "conditions": conds})
    return fams

def check_hard_rules(patients, relatives, visits, ranges=RANGES):
    """Returns {rule: number of failing rows}. Pass = every number is 0."""
    out = {}
    v = visits.merge(patients[["patient_id", "sex", "birth_year"]], on="patient_id", how="left")
    v = v.sort_values(["patient_id", "visit_no"])
    by_patient = v.groupby("patient_id")

    # 1. every value inside its dictionary min-max range (a missing column is a failure, not skipped)
    bad = pd.Series(False, index=v.index)
    for col, (lo, hi) in ranges.items():
        if col not in v.columns:
            out[f"1 column missing: {col}"] = 1
            continue
        bad |= v[col].notna() & ~v[col].between(lo, hi)
    out["1 value inside dictionary range"] = int(bad.sum())

    # 2. age matches birth year and visit date
    out["2 age matches birth year and visit date"] = int((v["age"] != v["visit_date"].dt.year - v["birth_year"]).sum())

    # 3. visit dates move forward
    out["3 visit dates move forward"] = int((by_patient["visit_date"].diff().dt.days <= 0).sum())

    # 4. pack-years never go down
    out["4 pack-years never go down"] = int((by_patient["pack_years"].diff() < 0).sum())

    # 5. a diagnosis, once recorded, stays
    out["5 diagnosis stays in later visits"] = int(sum((by_patient[f"has_{d}"].diff() < 0).sum() for d in DOMAIN))

    # 6. a relative's diagnosis falls inside their lifetime
    r = relatives.dropna(subset=["disease"])
    outside = (r["dx_year"] < r["birth_year"]) | (r["death_year"].notna() & (r["dx_year"] > r["death_year"]))
    out["6 relative diagnosis inside their lifetime"] = int(outside.sum())

    # 7. smoking status never goes back to "never"
    was_smoker = by_patient["smoking_status"].transform(lambda s: (s != "never").cummax().shift(fill_value=False))
    out["7 smoking never goes back to never"] = int((was_smoker & (v["smoking_status"] == "never")).sum())

    # 8. gestational diabetes only for females
    out["8 gestational diabetes only for females"] = int(((patients["sex"] == "Male") & (patients["gestational_diabetes"] == 1)).sum())

    # 9. parents older than patient; grandparents older than that parent; degree matches relation
    rel = relatives.drop_duplicates(["patient_id", "relative_no"]).merge(
        patients[["patient_id", "birth_year"]].rename(columns={"birth_year": "patient_by"}), on="patient_id")
    parents = rel[rel["relation"].isin(["father", "mother"])]
    out["9 parents older than patient"] = int((parents["birth_year"] >= parents["patient_by"]).sum())
    grand = rel[rel["relation"].str.contains("'s")].assign(parent=lambda d: d["relation"].str.split("'s").str[0])
    grand = grand.merge(parents[["patient_id", "relation", "birth_year"]].rename(
        columns={"relation": "parent", "birth_year": "parent_by"}), on=["patient_id", "parent"])
    out["9 grandparents older than parent"] = int((grand["birth_year"] >= grand["parent_by"]).sum())
    expected_degree = np.where(rel["relation"].str.contains("'s"), 2, 1)
    out["9 degree matches relation"] = int((rel["degree"] != expected_degree).sum())

    # 10. computed fields match their inputs
    def mismatch(col, expected, tol):
        has = v[col].notna() & expected.notna()
        return int((has & ((v[col] - expected).abs() > tol)).sum())
    out["10 computed field matches: bmi"] = mismatch("bmi", (v["weight_kg"] / (v["height_cm"] / 100) ** 2).round(1), 0.05)
    out["10 computed field matches: homa_ir"] = mismatch("homa_ir", (v["fpg"] * v["insulin"] / 405).clip(0.2, 25).round(1), 0.05)
    out["10 computed field matches: fev1_fvc"] = mismatch("fev1_fvc", (v["fev1"] / v["fvc"] * 100).round(1), 0.05)
    egfr = pd.Series([np.nan if pd.isna(c) else float(np.clip(round(egfr_ckd_epi(c, a, s)), 5, 140))
                      for c, a, s in zip(v["creatinine"], v["age"], v["sex"])], index=v.index)
    out["10 computed field matches: egfr"] = mismatch("egfr", egfr, 0.5)
    fams = _family(relatives)
    exp = patients.set_index("patient_id")[[c for c in patients if c.startswith("exp_")]]
    exposures = {pid: {k[4:]: bool(x) for k, x in row.items()} for pid, row in exp.iterrows()}
    for d in DOMAIN:
        expected = pd.Series([fhrs(fams.get(pid, []), exposures.get(pid, {}), d, dt.year)
                              for pid, dt in zip(v["patient_id"], v["visit_date"])], index=v.index)
        out[f"10 computed field matches: fhrs_{d}"] = mismatch(f"fhrs_{d}", expected, 0.005)
    return out

def print_report(results):
    failed = {k: n for k, n in results.items() if n > 0}
    for rule, n in results.items():
        print(f"{'PASS' if n == 0 else 'FAIL'}  {rule}" + (f"  ({n:,} rows)" if n else ""))
    print("\nALL HARD RULES PASS" if not failed else f"\n{len(failed)} RULE(S) FAILED")
    return not failed

In [ ]:
%%ipytest -q
# TEST: a small clean record passes; breaking one rule makes exactly that rule fail
import pytest

SMALL_RANGES = {"sbp": (70, 220), "bmi": (12.0, 60.0), "age": (18, 100)}

def clean():
    patients = pd.DataFrame({"patient_id": [0], "sex": ["Female"], "birth_year": [1980],
                             "gestational_diabetes": [0], "exp_secondhand_smoke": [0],
                             "exp_high_salt_diet": [0], "exp_processed_diet": [0],
                             "exp_low_home_activity": [0], "exp_indoor_mold": [0], "exp_coal_heating": [0]})
    relatives = pd.DataFrame({
        "patient_id": [0, 0, 0], "relative_no": [1, 2, 3],
        "relation": ["father", "mother", "father's father"], "degree": [1, 1, 2],
        "sex": ["M", "F", "M"], "birth_year": [1950, 1952, 1920], "death_year": [None, None, 1990],
        "disease": ["CVD", None, None], "dx_year": [2010, None, None], "dx_age": [60, None, None],
        "confirmed": [True, None, None]})
    visits = pd.DataFrame({
        "patient_id": [0, 0], "visit_no": [1, 2],
        "visit_date": pd.to_datetime(["2021-03-01", "2022-03-01"]), "age": [41, 42],
        "sbp": [120, 125], "weight_kg": [60.0, 61.0], "height_cm": [160.0, 160.0],
        "bmi": [23.4, 23.8], "fpg": [90, float("nan")], "insulin": [10.0, float("nan")],
        "homa_ir": [2.2, float("nan")], "fev1": [3.0, float("nan")], "fvc": [4.0, float("nan")],
        "fev1_fvc": [75.0, float("nan")], "creatinine": [0.8, float("nan")],
        "egfr": [round(egfr_ckd_epi(0.8, 41, "Female")), float("nan")],
        "pack_years": [0.0, 0.0], "smoking_status": ["never", "never"],
        "has_CVD": [0, 0], "has_T2D": [0, 0], "has_CRD": [0, 0], "has_NEURO": [0, 0],
        "fhrs_CVD": [0.5, 0.5], "fhrs_T2D": [0.0, 0.0], "fhrs_CRD": [0.0, 0.0], "fhrs_NEURO": [0.0, 0.0]})
    return patients, relatives, visits

def fails(p, r, v, ranges=SMALL_RANGES):
    return {k: n for k, n in check_hard_rules(p, r, v, ranges).items() if n > 0}

def test_clean_data_has_zero_failures():
    assert fails(*clean()) == {}

def test_rule_1_value_out_of_range():
    p, r, v = clean(); v.loc[1, "sbp"] = 300
    assert fails(p, r, v) == {"1 value inside dictionary range": 1}

def test_rule_1_missing_column_is_a_failure_not_skipped():
    p, r, v = clean()
    assert "1 column missing: hba1c" in fails(p, r, v, {**SMALL_RANGES, "hba1c": (4.0, 15.0)})

def test_rule_2_age_matches_birth_year():
    p, r, v = clean(); v.loc[1, "age"] = 50
    assert fails(p, r, v) == {"2 age matches birth year and visit date": 1}

def test_rule_3_visit_dates_move_forward():
    p, r, v = clean(); v.loc[1, "visit_date"] = pd.Timestamp("2020-01-01"); v.loc[1, "age"] = 40
    assert "3 visit dates move forward" in fails(p, r, v)

def test_rule_4_pack_years_never_go_down():
    p, r, v = clean(); v.loc[0, "pack_years"] = 5.0
    assert "4 pack-years never go down" in fails(p, r, v)

def test_rule_5_diagnosis_stays():
    p, r, v = clean(); v.loc[0, "has_CVD"] = 1
    assert "5 diagnosis stays in later visits" in fails(p, r, v)

def test_rule_6_relative_diagnosis_inside_their_life():
    p, r, v = clean(); r.loc[0, "death_year"] = 2005
    assert "6 relative diagnosis inside their lifetime" in fails(p, r, v)

def test_rule_7_smoking_never_goes_back_to_never():
    p, r, v = clean(); v.loc[0, "smoking_status"] = "former"
    assert "7 smoking never goes back to never" in fails(p, r, v)

def test_rule_8_gestational_diabetes_only_female():
    p, r, v = clean(); p.loc[0, "sex"] = "Male"; p.loc[0, "gestational_diabetes"] = 1
    assert "8 gestational diabetes only for females" in fails(p, r, v)

def test_rule_9_parents_older_and_degree_matches_relation():
    p, r, v = clean(); r.loc[0, "birth_year"] = 1990
    assert "9 parents older than patient" in fails(p, r, v)
    p, r, v = clean(); r.loc[2, "degree"] = 1
    assert "9 degree matches relation" in fails(p, r, v)

def test_rule_10_computed_fields_match_inputs():
    for col, bad in [("bmi", 30.0), ("homa_ir", 9.9), ("fev1_fvc", 50.0), ("egfr", 20), ("fhrs_CVD", 3.0)]:
        p, r, v = clean(); v.loc[0, col] = bad
        assert f"10 computed field matches: {col}" in fails(p, r, v), col

In [ ]:
# RUN  (about 30-60 seconds)
HARD_RULES = check_hard_rules(PATIENTS, RELATIVES, VISITS)
print_report(HARD_RULES)

## Cell 3: Population check
(next)

## Cell 4: Parameter recovery
(next)

## Cell 5: Doctor review sample
(next)